# 01 — API surfaces: from `messages.create` to the Tool Runner

One task domain (a tiny order shop, [`orders.json`](orders.json)), five ways to talk to Claude. Each step adds **one** thing the previous step could not do. Read top to bottom.

| # | Surface | Call | You own | Use when |
|---|---|---|---|---|
| 1 | Plain call | `client.messages.create` | everything | One request, one answer. The base of everything else. |
| 2 | Streaming | `client.messages.stream` | everything | Long output / UI. Same request, delivered incrementally. |
| 3 | Structured output | `client.messages.parse(output_format=Model)` | schema | You need **data**, not prose. Validated Pydantic object back. |
| 4 | Manual tool loop | `messages.create` + `while stop_reason == "tool_use"` | loop + tools | Learning, or you need control the runner's hooks don't give. |
| 5 | Tool Runner | `client.beta.messages.tool_runner` + `@beta_tool` | tools | **Default for custom-tool agents.** The SDK runs step 4 for you. |

All five are the **same endpoint** (`POST /v1/messages`) — `parse`, `stream` and `tool_runner` are SDK helpers on top of it. The next step up (Claude Agent SDK: built-in tools, subagents, hooks) is in [02/agent_sdk](../02_agentic_loop/agent_sdk/).

> **`messages.complete`?** Doesn't exist. What you may have seen is `client.completions.create` — the **legacy Text Completions API** (`"\n\nHuman: … \n\nAssistant:"` prompts). Don't use it; it's removed in SDK 1.x. Everything new is `messages.*`.

**Rule of thumb ("start simple")**: pick the lowest row that solves the problem.

## 0 · Setup

`config.client` / `config.settings` are the repo-wide client and model IDs (see root `config.py`). Run Jupyter from this folder (`uv run --with jupyter jupyter lab`) or select the repo `.venv` as kernel in VS Code.

In [ ]:
import json
from pathlib import Path

from config import client, settings

MODEL = settings.default_model
ORDERS = json.loads(Path("orders.json").read_text())
MODEL, list(ORDERS)

## 1 · `messages.create` — one request, one response

The API is **stateless**: you send the full `messages` list, you get back one assistant message. The response is a list of **content blocks** (text, thinking, tool_use, …) plus `stop_reason` and `usage`. Always branch on `block.type`.

In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    system="You are the support assistant of a small electronics shop. Be brief.",
    messages=[{"role": "user", "content": "A customer writes: 'My dock still hasn't arrived'. Draft a one-sentence reply."}],
)

for block in response.content:
    if block.type == "text":
        print(block.text)

print("\nstop_reason:", response.stop_reason)
print("usage:", response.usage.input_tokens, "in /", response.usage.output_tokens, "out")

**Multi-turn** = you append the assistant `content` and the next user turn yourself, then call again. Nothing is remembered server-side.

In [ ]:
messages = [
    {"role": "user", "content": "My name is Jane."},
]
r1 = client.messages.create(model=MODEL, max_tokens=256, messages=messages)
messages.append({"role": "assistant", "content": r1.content})  # append full content, not just text
messages.append({"role": "user", "content": "What's my name?"})
r2 = client.messages.create(model=MODEL, max_tokens=256, messages=messages)
print(r2.content[0].text)

## 2 · `messages.stream` — same request, incremental delivery

Identical parameters. Use it for long outputs (avoids HTTP timeouts on large `max_tokens`) and live UIs. `get_final_message()` gives you the same `Message` object as step 1.

In [ ]:
with client.messages.stream(
    model=MODEL,
    max_tokens=1024,
    messages=[{"role": "user", "content": "In three short bullets: why do parcels get delayed?"}],
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
    final = stream.get_final_message()

print("\n\nstop_reason:", final.stop_reason)

## 3 · `messages.parse` — data instead of prose

When your code consumes the answer, don't regex the text. Give a **Pydantic model** as `output_format`; the SDK turns it into a JSON schema (`output_config.format`), the API **constrains decoding** to that schema, and the SDK validates the result into `response.parsed_output`.

This replaces the old "force a tool call to get JSON" trick (which newer models reject: forced `tool_choice` returns 400 on Opus 5.5 / Sonnet 5.5).

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field


class Ticket(BaseModel):
    order_id: str | None = Field(description="Order ID like 'A-1002' if mentioned")
    intent: Literal["where_is_my_order", "refund", "complaint", "other"]
    sentiment: Literal["positive", "neutral", "negative"]
    summary: str = Field(description="One sentence")


email = "Hi, it's Jane. Order A-1002 (the dock) was due last week and it's STILL not here. This is really annoying."

parsed = client.messages.parse(
    model=MODEL,
    max_tokens=1024,
    messages=[{"role": "user", "content": f"Classify this support email:\n\n{email}"}],
    output_format=Ticket,
)

ticket: Ticket = parsed.parsed_output  # a validated Ticket instance
ticket

`parse` is still **one call, no tools**. Under the hood it's `messages.create(output_config={"format": {"type": "json_schema", "schema": ...}})` — use that form when you have a raw JSON schema instead of a Pydantic class.

Now the model needs information it doesn't have (the order status) → **tools**.

## 4 · Manual tool loop — what the Tool Runner automates

A tool is a name + description + JSON schema. Claude doesn't run it; it answers with `stop_reason == "tool_use"` and a `tool_use` block. **You** run the function, send back a `tool_result` (matched by `tool_use_id`), and call again — until Claude stops asking.

Read this once so step 5 isn't magic.

In [ ]:
def get_order(order_id: str) -> dict:
    if order_id not in ORDERS:
        raise KeyError(f"Unknown order '{order_id}'. Valid IDs look like 'A-1001'.")
    return ORDERS[order_id]


TOOLS = [
    {
        "name": "get_order",
        "description": "Look up one order by ID. Returns customer, item, status and ETA.",
        "input_schema": {
            "type": "object",
            "properties": {"order_id": {"type": "string", "description": "e.g. 'A-1002'"}},
            "required": ["order_id"],
        },
    }
]

messages = [{"role": "user", "content": "Where are orders A-1002 and A-9999?"}]

for turn in range(5):  # always cap the loop
    response = client.messages.create(model=MODEL, max_tokens=1024, tools=TOOLS, messages=messages)
    messages.append({"role": "assistant", "content": response.content})
    if response.stop_reason != "tool_use":
        break

    results = []
    for block in response.content:  # Claude may call several tools in parallel
        if block.type != "tool_use":
            continue
        print(f"turn {turn}: {block.name}({block.input})")
        try:
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(get_order(**block.input))})
        except Exception as e:  # errors are results, not crashes
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(e), "is_error": True})
    messages.append({"role": "user", "content": results})  # all results in ONE user message

print()
print(next(b.text for b in response.content if b.type == "text"))

Everything above that isn't `get_order` itself is boilerplate: schema by hand, the loop, the cap, appending, `tool_use_id` matching, error wrapping. That's exactly what the Tool Runner owns.

## 5 · `beta.messages.tool_runner` — the intended way to run custom tools

- `@beta_tool` builds the schema from **type hints + docstring** (the `Args:` section becomes parameter descriptions).
- The runner calls the API, runs your functions, appends results, loops until no more `tool_use`, respects `max_iterations`.
- Exceptions become `is_error: true` results automatically; raise `ToolError` for a clean message the model should read.

Same task as step 4:

In [ ]:
from anthropic import beta_tool
from anthropic.lib.tools import ToolError


@beta_tool
def get_order_tool(order_id: str) -> str:
    """Look up one order by ID. Returns customer, item, status and ETA.

    Args:
        order_id: Order ID, e.g. 'A-1002'.
    """
    if order_id not in ORDERS:
        raise ToolError(f"Unknown order '{order_id}'. Valid IDs look like 'A-1001'.")
    return json.dumps(ORDERS[order_id])


runner = client.beta.messages.tool_runner(
    model=MODEL,
    max_tokens=1024,
    max_iterations=5,
    tools=[get_order_tool],
    messages=[{"role": "user", "content": "Where are orders A-1002 and A-9999?"}],
)
final = runner.until_done()  # just the last message
print(next(b.text for b in final.content if b.type == "text"))

### 5b · Iterate instead of `until_done()` — your per-turn hook

The runner is an iterable: each iteration yields one assistant message. That's where logging, approval gates, or early `break` go — without writing the loop yourself.

In [ ]:
runner = client.beta.messages.tool_runner(
    model=MODEL,
    max_tokens=1024,
    max_iterations=5,
    tools=[get_order_tool],
    messages=[{"role": "user", "content": "Which of A-1001, A-1002, A-1003 are not delivered yet?"}],
)

for i, message in enumerate(runner):
    calls = [f"{b.name}({b.input})" for b in message.content if b.type == "tool_use"]
    print(f"iteration {i}: stop_reason={message.stop_reason} calls={calls}")
    tool_response = runner.generate_tool_call_response()  # the tool_result message the runner will send next (or None)
    if tool_response is not None:
        errors = [b for b in tool_response["content"] if b.get("is_error")]
        if errors:
            print("  tool errors:", errors)

print()
print(next(b.text for b in message.content if b.type == "text"))

### 5c · Tools **and** structured output

`tool_runner` takes the same `output_format` as `parse`: tools during the loop, a validated object at the end. This is the typical production shape: *"agent gathers facts → returns data your code can use"*.

In [ ]:
class OrderReport(BaseModel):
    delayed_order_ids: list[str]
    customer_message: str = Field(description="One friendly sentence for the customer")


runner = client.beta.messages.tool_runner(
    model=MODEL,
    max_tokens=1024,
    max_iterations=5,
    tools=[get_order_tool],
    output_format=OrderReport,
    messages=[{"role": "user", "content": "Jane owns A-1001 and A-1002. Check both and report."}],
)
report: OrderReport = runner.until_done().parsed_output
report

## Decision guide

```
Do you need Claude to call your code / fetch data?
├─ no ─ need machine-readable output? ── yes → messages.parse
│                                    └─ no  → messages.create  (stream if long / UI)
└─ yes ─ only your own tools?  ──────── yes → tool_runner (+ output_format for data)
                                               manual loop only if the runner's per-turn hooks don't fit
```

| | create | parse | manual loop | tool_runner |
|---|---|---|---|---|
| Package | `anthropic` | `anthropic` | `anthropic` | `anthropic` (beta) |
| Calls per run | 1 | 1 | n | n |
| Tools | — | — | yours | yours |
| Loop owned by | — | — | you | SDK |
| Typed output | — | ✅ | DIY | ✅ `output_format` |

**Common mix-ups**
- *`parse` ≠ tool use.* `parse` constrains the **final answer**; tools let Claude **fetch** things mid-answer. Combine them via `tool_runner(output_format=...)`.
- *Forcing a tool to get JSON* (`tool_choice={"type": "tool", ...}`) is the old pattern — use `parse`.

## Sources
- Intro / overview — <https://platform.claude.com/docs/en/intro>
- Tool Runner — <https://platform.claude.com/docs/en/agents-and-tools/tool-use/tool-runner>
- Handle tool calls (manual loop) — <https://platform.claude.com/docs/en/agents-and-tools/tool-use/handle-tool-calls>
- Structured outputs — <https://platform.claude.com/docs/en/build-with-claude/structured-outputs>
- Streaming — <https://platform.claude.com/docs/en/build-with-claude/streaming>